# Reproduce everything

**Reconstruction-independent moment tests for quantum-computed dynamical spectra** — master notebook.

Every result that is not device data is an exact classical computation (`numpy`/`scipy`/`mpmath`; the
re-analysis of the retained `ibm_fez` counts and the noise-model forecasts also use `qiskit`/`qiskit-aer`).
This notebook drives the scripts in `src/`, each in its own process, exactly as the `Makefile` does
(`cd src && python <script>.py`). They write JSON into `data/` and the figure data into `paper/figs/*.dat`,
so `git status` afterwards shows whether a committed number moved. It includes the 2026-09-27 (revision
phase A) scripts and the corrected ones, so `make reproduce` needs nothing else.

The manuscript in `paper/` is being revised for SciPost Physics Core; where the text and a data file
disagree, the data file is the reference. See [`docs/REPRODUCE.md`](../docs/REPRODUCE.md) for the figure →
script → data map, the key numbers and the known gaps, and [`docs/FILE_INDEX.md`](../docs/FILE_INDEX.md)
for every file.

Run time on the authors' laptop: about 45–60 min for the default cells (the theory numerics take about
15 min of that). The device reference Monte Carlo (7–20 min, needs `qiskit`) is switched off by default;
set `RUN_DEVICE_MC = True` below to include it.

In [ ]:
import os, subprocess, sys, time, pathlib

# The kernel may start in the repository root (python / papermill) or in notebooks/ (Jupyter, nbconvert,
# `make reproduce`): walk up from the working directory to the folder that holds src/spectral_lanczos.py.
ROOT = pathlib.Path.cwd().resolve()
for _cand in (ROOT, *ROOT.parents):
    if (_cand / "src" / "spectral_lanczos.py").exists():
        ROOT = _cand
        break
else:
    raise RuntimeError("open this notebook from the repository or from its notebooks/ folder")
SRC = ROOT / "src"
print("repository root:", ROOT)

RUN_DEVICE_MC = False          # delta0_reference_mc.py: 7-20 min, needs qiskit (no QPU)
FAILED = []

def run(script, *args, tail=2500):
    # Run one script in a fresh process from src/, as the Makefile does; print the tail of its output.
    cmd = [sys.executable, script, *args]
    print("$ cd src && python " + " ".join([script, *args]))
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=SRC,
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"}, encoding="utf-8", errors="replace")
    if r.stdout:
        print(r.stdout[-tail:])
    if r.returncode:
        FAILED.append(" ".join([script, *args]))
        print("!! exit code", r.returncode, "\n!! STDERR:", r.stderr[-2000:])
    print(f"-- {time.time() - t0:.0f} s " + "-" * 60)
    return r.returncode

## 1 · Fast reproducibility check (seconds)

`verify.py`: the current-probe moments of the doped `L=6, U/t=4` ring, the shot-budget thresholds
`τ_k = z·δ_k` at `N_s = 5×10⁴` (the per-circuit budget of the companion's `ibm_fez` run) with an assumed 2%
bias budget, and one truncation (`d = 97`, deterministic `lexsort` order) that the lone first moment passes
and the second moment rejects: `|Δm₁| = 0.186 < 0.229`, `|Δm₂| = 2.729 > 1.848`. One catch, not a
guarantee of power (section 2 lists the truncations the whole battery passes).

In [ ]:
run("verify.py")

## 2 · The interval-moment tests (deterministic truncation order, ≈3 min)

`interval_moment.py` (lone `m₁` on the 22-point grid), `interval_battery.py` (the joint `(m₀,m₁,m₂)`
battery for every truncation depth at 0/2/4% bias, the degenerate `|ψ₀|²` shells and the tie-break
sensitivity; ≈2 min) and `interval_moment_mc.py` (sampled shots at `N_s = 5×10⁴`; writes
`paper/figs/momentmc_*.dat`). All three are exact-state forecasts: no moment was measured on a device. At the
deployed 2% bias the whole battery still passes 12 truncations.

In [ ]:
for s in ["interval_moment.py", "interval_battery.py", "interval_moment_mc.py"]:
    run(s)

## 3 · Theory numerics, within-sector redistributions and the Christoffel bounds (≈20 min)

`small_checks.py --all` rebuilds every key of `data/2026-09-27_theory_numerics.json`: it runs
`within_sector_lp.py`, `christoffel_tolerance_lp.py`, `necessary_sufficient_composition.py` (two
necessary conditions, Hankel and Hausdorff, side by side), `estimator_form_offeigenstate.py`,
`run_inverse_moment_falsifier.py --r8` and `export_christoffel_dat.py` (which also writes
`christoffel_*.dat`), then its own keys (≈15 min). `within_sector_control.py` is the 2026-08-24 random
redistribution record; `markov_krein_window.py` the two-sided window bound; `export_bracketing_dat.py` the
Gauss–Radau bracketing data (its `bracket_maxW.dat` holds grid maxima that no figure reads; the exact
maxima are in `christoffel_maxW.dat`); `chigap_check.py` the Christoffel empty-region check.

In [ ]:
run("small_checks.py", "--all")
for s in ["within_sector_control.py", "markov_krein_window.py", "export_bracketing_dat.py", "chigap_check.py"]:
    run(s)

## 4 · The demonstrations across observables (exact simulation)

The current-response sum rules (`m₀` blind, `m₁` detects), the collective channels, the negative-order
(`m₋₁`) test, the teeth at `L=6` (independent estimator against a circular control) and at `L=12` on the
shared subspace state, and the U(1) Gauss-law state-level check (the bit-flip sweep, and the ε-admixture
curve that `fig_gausslaw` plots).

Two reproducibility notes. `run_teeth_shared.py` orders the truncation with `argsort`; a rerun reproduces
the two 5% crossings (`d ≈ 15977` and `43554`) but moves the three smallest-`d` points of
`paper/figs/teeth_shared.dat` (up to 6% in `m̄₁` at `d = 180`) against the committed 2026-08-26 record, so
`git status` will show that file. `run_inverse_moment_falsifier.py` (without `--r8`, ≈9 min) rewrites
`data/2026-08-28_inverse_moment_falsifier.json` and `inverse_falsifier_*.dat`, which it reproduces bit for
bit; it reports relative shifts, and the noise-normalized comparison (`--r8`, run by `small_checks.py`)
shows that `m₋₁` fires after `m₁` and `m₂` for the deployed error.

In [ ]:
for s in ["run_sumrule_falsifier.py", "run_collective_sumrule_falsifier.py",
          "run_inverse_moment_falsifier.py", "run_falsifier_teeth.py",
          "run_teeth_shared.py", "run_gausslaw_falsifier.py", "export_gauss_state.py"]:
    run(s)

## 5 · The blinded, pre-registered test (sealed) and its post hoc addenda

The seal (`data/prereg.sha256`) is checked before anything is read. `blind_score.py` re-scores the sealed
verdicts: TPR 101/188 = 53.7% at FPR 1/112; truncation 56/56, spurious 45/61, Krylov 0/71; two sealed
secondary predictions failed (one-node Krylov 0/15; `m₀`-preserving spurious 20/34). `blind_addenda.py`
adds the post hoc analyses (modelled FPR with and without bias, an exact rebuild of the 56 sealed
truncations from the stored tie order, a post hoc shared-state rescoring, Krylov residuals, the calibrated
scope); it aborts and writes nothing if the rebuild does not reproduce the sealed record.
`separating_counts.py` recounts `fig_separating` from the sealed record and writes `sep_*.dat`.

In [ ]:
for s in ["blind_score.py", "blind_addenda.py", "separating_counts.py"]:
    run(s)

## 6 · Figure data from the committed caches and records

The `export_*` scripts write `paper/figs/*.dat` (and the `A(k,ω)`/`S(q,ω)` rasters) from
`src/cache/*_L12.npz`, `data/heron_spectral.json` (`fig_heron`) and
`data/2026-09-27_delta0_reference_mc.json` (`fig_device`); `check_sqw_extent.py` records what the `fig_sqw`
rasters draw below the charge-onset line and what the display windows clip. `export_momentcone_dat.py` takes ≈2 min. The
device reference Monte Carlo that produces the `fig_device` JSON runs only if `RUN_DEVICE_MC` is set.

In [ ]:
if RUN_DEVICE_MC:
    run("delta0_reference_mc.py")
for s in ["export_akw_dat.py", "export_sqw_dat.py", "check_sqw_extent.py", "export_teeth_dat.py",
          "export_momentcone_dat.py", "export_heron_dat.py", "export_device_dat.py"]:
    run(s)

## 7 · Optional: noise-model forecasts and the retained `ibm_fez` counts (`qiskit`, `qiskit-aer`)

Not needed for the figures (their outputs are committed under `data/`). Uncomment to run.

- `nk_falsifier.py` — the momentum-distribution `n_k` forecast (≈9% of the Fermi step, FT-only).
- `offdiag_noise_forecast.py`, `offdiag_gsurface.py` — the off-diagonal depolarizing-bias floor (≈37–116% of the signal at `ibm_fez` depth; dated outputs).
- `bond_moment_estimator.py`, `estimator_scaling.py` — the bond-basis estimator in noiseless simulation (≈27 min at the default L=6; `BME_L=4` ≈3 min) and its circuit counts (≈3 min).
- `joint_covariance_composition.py`, `lever1_flip_experiment.py` — the same-shot covariance analysis (simulation only; no power gain).
- `aer_noiseless_baseline.py`, `bootstrap_device_curve.py`, `retention_matched_control.py` — re-analysis of the retained `ibm_fez` counts.
- `hardware_matched_job_L8.py` — the device-side job (**use your own IBM account; never commit a token**). `hardware_blind_job.py` was prepared and never run; its state preparation is a placeholder.

In [ ]:
# run("nk_falsifier.py")
# run("offdiag_noise_forecast.py")
# run("bond_moment_estimator.py")
# run("retention_matched_control.py")
print("optional cells: uncomment above to run")

## 8 · Summary and the manuscript

`make paper` compiles `paper/main.tex` with the committed `main.bbl`. The cell below lists any script that
failed and stops with an error if one did, so `make reproduce` fails loudly.

In [ ]:
if FAILED:
    raise RuntimeError(f"{len(FAILED)} script(s) failed: {FAILED}")
print("all scripts ran; 'git status data paper/figs' shows any committed number that moved")